# AIE26 第2回 P3：Epoch × rank × Seed の分析

27条件（Epoch=5/10/20、rank=8/16/32、Seed=42/123/2026）の独立実行結果を再計算・可視化する。

**対象**：形式遵守率、内容正解率、Validation Loss、最終Training Loss、学習時間、質問別の誤答。全条件で同一の21問を使用するため、平均正解率の差を独立な新規テストデータでの性能差と解釈しない。

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

HERE = Path.cwd()
if not (HERE / 'summary.csv').is_file():
    HERE = Path('lectures/02/experiments/P3')
assert (HERE / 'summary.csv').is_file(), 'Run notebook from the P3 directory or repository root.'

df = pd.read_csv(HERE / 'summary.csv')
assert len(df) == 27 and df['status'].eq('completed').all()
assert set(df['epochs']) == {5, 10, 20}
assert set(df['rank']) == {8, 16, 32}
assert set(df['seed']) == {42, 123, 2026}
assert not df.duplicated(['epochs','rank','seed']).any()
print('Completed conditions:', len(df))
print('Format compliance:', df['format_rate'].min(), df['format_rate'].max())
display(df.head())

In [ ]:
cols = ['content_rate','format_rate','training_seconds','validation_loss','last_logged_training_loss']
g = df.groupby(['epochs','rank'])[cols].agg(['mean','std','median'])
display(g.round(4))
# 既存の集計表と照合
check = pd.read_csv(HERE / 'group_summary.csv')
for row in check.itertuples():
    got = g.loc[(row.epochs, row.rank), ('content_rate','mean')]
    assert np.isclose(row.content_rate_mean, got)
print('Aggregate consistency: OK')

In [ ]:
def heatmap(metric, title, scale=1., fmt='.1f'):
    pivot = df.pivot_table(index='epochs', columns='rank', values=metric, aggfunc='mean')
    values = pivot.to_numpy() * scale
    fig, ax = plt.subplots(figsize=(6.5, 3.8))
    im = ax.imshow(values, cmap='viridis', aspect='auto')
    ax.set_xticks(range(3), pivot.columns)
    ax.set_yticks(range(3), pivot.index)
    ax.set_xlabel('LoRA rank'); ax.set_ylabel('Epoch')
    ax.set_title(title)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, format(values[i,j],fmt), ha='center', va='center',
                    color='white' if values[i,j] < (values.min()+values.max())/2 else 'black')
    fig.colorbar(im, ax=ax)
    plt.show()

heatmap('content_rate', 'Content correctness: mean over 3 seeds (%)', 100)
heatmap('validation_loss', 'Validation loss: mean over 3 seeds', 1., '.3f')
heatmap('training_seconds', 'Training time: mean over 3 seeds (s)', 1., '.1f')

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
for r in sorted(df['rank'].unique()):
    d = df[df['rank']==r].groupby('epochs')['content_rate']
    x = np.array(sorted(df['epochs'].unique()))
    y = d.mean().reindex(x).to_numpy() * 100
    sd = d.std().reindex(x).to_numpy() * 100
    ax.errorbar(x, y, yerr=sd, marker='o', capsize=4, label=f'rank {r}')
ax.set_xticks([5,10,20]);ax.set_xlabel('Epoch');ax.set_ylabel('Content correctness (%)')
ax.set_title('Epoch × rank interaction (bars = seed SD)')
ax.grid(alpha=0.25);ax.legend();plt.show()

In [ ]:
# 3 Seed平均の正解率と時間から効率を比較する
s = df.groupby(['epochs','rank']).agg(
    accuracy=('content_rate','mean'),
    val_loss=('validation_loss','mean'),
    train_s=('training_seconds','mean'),
    median_train_s=('training_seconds','median'),
).reset_index()
s['accuracy_pct'] = (s['accuracy']*100).round(2)
s['correct_of_63'] = (s['accuracy']*63).round().astype(int)
display(s.sort_values('accuracy',ascending=False).round(3))
print('Do not treat 63 as 63 independent test questions: the same 21 questions are repeated across seeds.')

In [ ]:
errors = HERE / 'question_errors.csv'
if errors.is_file():
    q = pd.read_csv(errors)
    print('Question-level errors:',len(q))
    display(q.groupby(['epochs','rank','error_type']).size().unstack(fill_value=0))
    display(q.groupby('question').size().sort_values(ascending=False).head(15).rename('error_count'))
    display(q.head(12))
else:
    print('No question_errors.csv; question-level evaluation not available in source JSON.')

## 解釈上の注意

- 形式遵守率は全27条件で100%でも、内容正解率まで100%という意味ではない。
- Epoch・rankの効果と交互作用は**記述的**な傾向。各条件3 Seed・同一21問なので、統計的な優劣や因果的な過学習を断定しない。
- 検証Lossを使った設定選択と、未知テストデータでの最終評価を分離するのが望ましい。
- P3はウォームアップ・alphaの条件がP0〜P2と異なるため、実験系列を混同しない。